# Part I: Full Fine-Tuning (FFT) — XLM-RoBERTa Baseline
**Course:** Natural Language Processing — Alexandria University  
**Objective:** Full fine-tuning of XLM-RoBERTa with a causal LM head on the `flytech/python-codes-25k` dataset.

In [ ]:
# Cell 1: Install all required packages
!pip install -q transformers datasets accelerate bitsandbytes wandb
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118



In [ ]:
pip install --upgrade wandb

In [ ]:
# Cell 2: Imports
import torch
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import load_dataset

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


In [ ]:
# Cell 3: Load dataset and cap at 10,000 samples
print("Loading dataset...")
full_dataset = load_dataset("flytech/python-codes-25k", split="train")
print(f"Full dataset size : {len(full_dataset)} samples")
print(f"Columns           : {full_dataset.column_names}")
print(f"\nSample record:\n{full_dataset[0]}")

# ── Cap at 10,000 ──────────────────────────────────────────────────
SAMPLE_SIZE = 10_000
dataset = full_dataset.select(range(SAMPLE_SIZE))
print(f"\nUsing {len(dataset)} samples for Part I (FFT baseline)")

# ── Train / Eval split (95 / 5) ────────────────────────────────────
split = dataset.train_test_split(test_size=0.05, seed=42)
train_data = split["train"]   # ~9,500 samples
eval_data  = split["test"]    #   ~500 samples
print(f"Train : {len(train_data)} | Eval : {len(eval_data)}")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

Full dataset size : 49626 samples
Columns           : ['output', 'instruction', 'input', 'text']

Sample record:
{'output': "```python\ntasks = []\nwhile True:\n    task = input('Enter a task or type 'done' to finish: ')\n    if task == 'done': break\n    tasks.append(task)\nprint(f'Your to-do list for today: {tasks}')\n```", 'instruction': 'Help me set up my daily to-do list!', 'input': 'Setting up your daily to-do list...', 'text': "Help me set up my daily to-do list! Setting up your daily to-do list... ```python\ntasks = []\nwhile True:\n    task = input('Enter a task or type 'done' to finish: ')\n    if task == 'done': break\n    tasks.append(task)\nprint(f'Your to-do list for today: {tasks}')\n```"}

Using 10000 samples for Part I (FFT baseline)
Train : 9500 | Eval : 500


In [ ]:
# Cell 4: Load tokenizer and XLM-RoBERTa as a Causal LM
MODEL_NAME = "FacebookAI/xlm-roberta-base"  # 0.3B — fits on T4

print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# XLM-RoBERTa uses <s> as BOS but has no dedicated EOS as pad;
# set pad = eos to avoid padding issues in CLM training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Force encoder → decoder mode ──────────────────────────────────
# RoBERTa is encoder-only; setting is_decoder=True + attaching
# an LM head turns it into a language model (XLMRobertaForCausalLM)
config = AutoConfig.from_pretrained(MODEL_NAME)
config.is_decoder = True          # makes self-attention causal (masked)

# Load model from checkpoint if available, otherwise from base
if 'resume_from_checkpoint_path' in globals() and resume_from_checkpoint_path:
    print(f"Loading model from checkpoint: {resume_from_checkpoint_path}")
    model = AutoModelForCausalLM.from_pretrained(
        resume_from_checkpoint_path,
        config=config,
        torch_dtype=torch.bfloat16,
    )
else:
    print(f"Loading model from base: {MODEL_NAME}")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        config=config,
        torch_dtype=torch.bfloat16,
        ignore_mismatched_sizes=True,  # LM head is new — weights don't match
    )

model.config.use_cache = False    # required when gradient_checkpointing=True

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}  (FFT = 100% trainable)")
print(f"Model class         : {model.__class__.__name__}")

Loading model: FacebookAI/xlm-roberta-base
Loading model from checkpoint: /content/drive/MyDrive/NLP_Assignment4/roberta-fft-checkpoints/checkpoint-1000


HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/NLP_Assignment4/roberta-fft-checkpoints/checkpoint-1000'. Use `repo_type` argument if needed.

In [ ]:
# Cell 5: Tokenize — combine instruction + response into one sequence
MAX_LENGTH = 512  # ✅ kept at 512 to reduce training time on Colab

def tokenize_fn(examples):
    texts = []
    for instr, out in zip(examples["instruction"], examples["output"]):
        instr = instr or ""
        out   = out   or ""
        # Standard prompt template — will be reused for inference
        text = (
            f"### Instruction:\n{instr}\n\n"
            f"### Response:\n{out}"
        )
        texts.append(text)

    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    # In CLM, labels = input_ids (predict every next token)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizing train split...")
tokenized_train = train_data.map(
    tokenize_fn, batched=True, remove_columns=train_data.column_names
)
print("Tokenizing eval split...")
tokenized_eval = eval_data.map(
    tokenize_fn, batched=True, remove_columns=eval_data.column_names
)

tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")

print(f"\nTrain tokens shape: {tokenized_train[0]['input_ids'].shape}")
print(f"Eval  tokens shape: {tokenized_eval[0]['input_ids'].shape}")

Tokenizing train split...
Tokenizing eval split...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Train tokens shape: torch.Size([512])
Eval  tokens shape: torch.Size([512])


In [ ]:
# Cell 6: Training Arguments
# ── Warmup recalculation ──────────────────────────────────────────
# train samples = 9500, batch = 4, accum = 1 → steps/epoch = 2375
# 2 epochs → total_steps = 4750 → 10% warmup ≈ 475 steps
# ✅ FIXED: original code had warmup_steps=2350 (≈ 1 full epoch — wrong!)

TOTAL_STEPS  = (len(train_data) // 4) * 2   # ~4750
WARMUP_STEPS = int(0.10 * TOTAL_STEPS)       # ~475
print(f"Total training steps : {TOTAL_STEPS}")
print(f"Warmup steps (10%)   : {WARMUP_STEPS}")

training_args = TrainingArguments(
    output_dir                  = "./roberta_fft_output",

    # ── Batch / accumulation ───────────────────────────────────────
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,

    # ── Schedule ───────────────────────────────────────────────────
    num_train_epochs            = 2,
    learning_rate               = 5e-5,
    lr_scheduler_type           = "cosine",
    warmup_steps                = WARMUP_STEPS,   # ✅ FIXED

    # ── Optimizer & precision ──────────────────────────────────────
    optim                       = "adamw_torch",
    bf16                        = True,

    # ── Memory saving ─────────────────────────────────────────────
    gradient_checkpointing      = True,

    # ── Logging / saving ──────────────────────────────────────────
    eval_strategy               = "epoch",   # or "evaluation_strategy" on older HF
    save_strategy               = "epoch",
    logging_steps               = 50,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",

    report_to                   = "wandb",    # change to "wandb" if configured
)

Total training steps : 4750
Warmup steps (10%)   : 475


In [ ]:
# Cell 7: Data Collator + Trainer
# mlm=False → Causal LM (next-token prediction), NOT masked LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized_train,
    eval_dataset    = tokenized_eval,
    data_collator   = data_collator,
    processing_class= tokenizer,   # replaces deprecated `tokenizer=` arg
)

print("Trainer ready.")
print(f"  Train batches/epoch : {len(trainer.get_train_dataloader())}")
print(f"  Eval  batches       : {len(trainer.get_eval_dataloader())}")

Trainer ready.
  Train batches/epoch : 2375
  Eval  batches       : 63


In [ ]:
# Cell 8: Run training + Save to Google Drive
# Google Drive is mounted, GDRIVE_SAVE_PATH and resume_from_checkpoint_path are defined in an earlier cell.
resume_from_checkpoint_path = "/content/drive/MyDrive/NLP_Assignment4/roberta-fft-checkpoints/checkpoint-4800"
# ── Train ─────────────────────────────────────────────────────────
print("Starting FFT training...")
print("Watch: train_loss should drop from ~4.0 → ~1.5 by epoch 2\n")

train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint_path)
GDRIVE_SAVE_PATH ="/content/drive/MyDrive/NLP_Assignment4/roberta_fft_final"
print("\n── Training Summary ──────────────────────────────────────")
print(f"  Final train loss : {train_result.training_loss:.4f}")
print(f"  Total steps      : {train_result.global_step}")
metrics = trainer.evaluate()
print(f"  Final eval loss  : {metrics['eval_loss']:.4f}")
print("──────────────────────────────────────────────────────────")

# ── Save locally first (fast), then copy to Drive ─────────────────
LOCAL_SAVE_PATH = "./roberta_fft_final"
print(f"\nSaving model locally to  : {LOCAL_SAVE_PATH}")
trainer.save_model(LOCAL_SAVE_PATH)
tokenizer.save_pretrained(LOCAL_SAVE_PATH)

print(f"Copying to Google Drive  : {GDRIVE_SAVE_PATH}")
for filename in os.listdir(LOCAL_SAVE_PATH):
    src  = os.path.join(LOCAL_SAVE_PATH, filename)
    dest = os.path.join(GDRIVE_SAVE_PATH, filename)
    shutil.copy2(src, dest)

# ── Verify ────────────────────────────────────────────────────────
print("\n── Files saved to Google Drive ──────────────────────────")
for f in sorted(os.listdir(GDRIVE_SAVE_PATH)):
    size_mb = os.path.getsize(os.path.join(GDRIVE_SAVE_PATH, f)) / 1e6
    print(f"  {f:45s}  {size_mb:6.1f} MB")
print("──────────────────────────────────────────────────────────")
print("✅ Done — model is safe in Drive even if Colab disconnects.")

Starting FFT training...
Watch: train_loss should drop from ~4.0 → ~1.5 by epoch 2



There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias', 'roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.Layer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 9: Save model + tokenizer
SAVE_PATH = "./roberta_fft_final"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to: {SAVE_PATH}")

# List saved files
import os
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"  {f:40s}  {size:.1f} MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./roberta_fft_final
  generation_config.json                    0.0 MB
  tokenizer.json                            16.8 MB
  training_args.bin                         0.0 MB
  tokenizer_config.json                     0.0 MB
  config.json                               0.0 MB
  model.safetensors                         556.6 MB


In [ ]:
# Cell 10: Inference Test
# ─────────────────────────────────────────────────────────────────
# EXPECTED OUTPUT ANALYSIS (compare your actual output to this):
#
# XLM-RoBERTa is an ENCODER model repurposed as a CLM.
# After FFT on 9500 code samples for 2 epochs you should expect:
#
#  ✅ GOOD signs:
#     - Output begins with recognizable Python syntax (def, return, for)
#     - The ### Response: section contains code-like text
#     - No random unicode garbage (model learned the domain)
#
#  ⚠️  KNOWN LIMITATIONS of this approach:
#     - Output may repeat tokens or truncate mid-function
#     - Indentation or brackets may be wrong
#     - RoBERTa's causal mask approximation is weaker than GPT-style
#     - Eval loss ~1.5–2.5 is normal (not as low as a native decoder)
#
#  ❌ RED FLAGS (something went wrong if you see these):
#     - Completely random text / no Python keywords at all
#     - Output identical to prompt (model just copied input)
#     - RuntimeError about cache (set use_cache=False was missed)
# ─────────────────────────────────────────────────────────────────

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)

# ── Test prompts ─────────────────────────────────────────────────
test_prompts = [
    "Write a Python function to reverse a string",
    "Write a Python function to check if a number is prime",
    "Write a Python function that returns the factorial of n",
]

for prompt_text in test_prompts:
    print("=" * 60)
    print(f"📌 INSTRUCTION: {prompt_text}")
    print("-" * 60)

    full_prompt = (
        f"### Instruction:\n{prompt_text}\n\n"
        f"### Response:\n"
    )

    inputs = tokenizer(
        full_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,       # leave room for generation
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens    = 150,
            do_sample         = True,
            temperature       = 0.7,
            top_p             = 0.9,
            repetition_penalty= 1.2,   # reduces looping/repetition
            pad_token_id      = tokenizer.pad_token_id,   # ✅ FIXED: silences warning
            eos_token_id      = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (not the prompt)
    prompt_len     = inputs["input_ids"].shape[1]
    generated_ids  = outputs[0][prompt_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(generated_text)
    print()

In [ ]:
# Cell 11: Print loss history for your report
print("── Loss Log ────────────────────────────────────────────────")
log_history = trainer.state.log_history

print(f"{'Step':>6}  {'Train Loss':>12}  {'Eval Loss':>12}")
print("-" * 36)
for entry in log_history:
    step       = entry.get("step", "")
    train_loss = entry.get("loss",       None)
    eval_loss  = entry.get("eval_loss",  None)
    if train_loss or eval_loss:
        tl = f"{train_loss:.4f}" if train_loss else "    —   "
        el = f"{eval_loss:.4f}"  if eval_loss  else "    —   "
        print(f"{step:>6}  {tl:>12}  {el:>12}")

print("\n── Theoretical Expectations ────────────────────────────────")
print("  Epoch 1 eval_loss  →  typically 1.8 – 2.5")
print("  Epoch 2 eval_loss  →  typically 1.4 – 2.0")
print("  (Higher than GPT-style models because RoBERTa's")
print("   causal mask is an approximation on an encoder arch)")